# LeetCode 2090 — K Radius Subarray Averages bằng Python

Link đề: https://leetcode.com/problems/k-radius-subarray-averages/ · Độ khó: **Medium** · Chủ đề: Array, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def getAverages(self, nums: List[int], k: int) -> List[int]:
        ...
```

Lộ trình:
1. **Brute force**: với mỗi tâm `i`, cộng lại `2k + 1` phần tử — `O(n·k)`.
2. **Prefix sum**: tổng cửa sổ `= P[i+k+1] − P[i−k]` — `O(n)`.
3. **Cửa sổ trượt** độ dài cố định `2k + 1`: thêm phần tử mới, bớt phần tử cũ — `O(n)`, `O(1)` bộ nhớ phụ.
4. **Viết gọn bằng `accumulate` + lát cắt** — cùng ý tưởng prefix, ít vòng lặp Python.

## 1. Mục tiêu học

1. Chuyển "trung bình bán kính `k` quanh `i`" thành **tổng cửa sổ độ dài cố định `w = 2k + 1`** chia nguyên cho `w`.
2. Xác định đúng **tâm hợp lệ**: `k ≤ i ≤ n − 1 − k`; các vị trí khác là `-1` — kể cả khi `w > n` (không tâm nào hợp lệ).
3. Thành thạo hai kỹ thuật tổng đoạn `O(1)`: **prefix sum** và **cửa sổ trượt**, và biết chỗ đặt chỉ số `+1`/`-1`.
4. Hiểu phép chia nguyên của đề ("cắt về 0") và vì sao `//` của Python khớp ở đây (số không âm) nhưng **không** khớp khi có số âm.
5. Kiểm chứng bằng brute force và test các biên `k = 0`, `2k + 1 = n`, `2k + 1 > n`.

## 2. Đề bài & ví dụ

Cho mảng `nums` (`n` phần tử) và số nguyên `k`. **Trung bình bán kính `k`** tại tâm `i` là trung bình (chia nguyên, cắt phần thập phân) của mọi phần tử từ chỉ số `i − k` tới `i + k`. Nếu bên trái hoặc bên phải `i` có **ít hơn `k`** phần tử thì giá trị là `-1`. Trả về mảng `avgs` độ dài `n`.

**Ví dụ 1:** `nums = [7,4,3,9,1,8,5,2,6]`, `k = 3` → `[-1,-1,-1,5,4,4,-1,-1,-1]` (ví dụ `avgs[3] = 37 // 7 = 5`).

**Ví dụ 2:** `nums = [100000]`, `k = 0` → `[100000]`.

**Ví dụ 3:** `nums = [8]`, `k = 100000` → `[-1]`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ nums[i], k ≤ 10⁵`.

In [1]:
import random
import time
from collections import deque
from itertools import accumulate
from typing import List

examples = [(([7, 4, 3, 9, 1, 8, 5, 2, 6], 3), [-1, -1, -1, 5, 4, 4, -1, -1, -1]),
            (([100000], 0), [100000]),
            (([8], 100000), [-1])]
for (nums, k), want in examples:
    print(f"nums = {nums}, k = {k} -> {want}")

nums = [7, 4, 3, 9, 1, 8, 5, 2, 6], k = 3 -> [-1, -1, -1, 5, 4, 4, -1, -1, -1]
nums = [100000], k = 0 -> [100000]
nums = [8], k = 100000 -> [-1]


## 3. Kiến thức nền

### 3.1 Chia nguyên trong Python

`a // b` là chia **làm tròn xuống** (floor). Với `a ≥ 0`, `b > 0` thì floor = cắt về 0, đúng yêu cầu đề. Với số âm, `-7 // 2 = -4` còn "cắt về 0" cho `-3` — phải dùng `int(a / b)` hoặc xử lý dấu riêng (xem biến thể ở mục 15).

In [2]:
print("11 // 4 =", 11 // 4, "| 37 // 7 =", 37 // 7)
print("-7 // 2 =", -7 // 2, "| cắt về 0: int(-7 / 2) =", int(-7 / 2), "| -(7 // 2) =", -(7 // 2))

11 // 4 = 2 | 37 // 7 = 5
-7 // 2 = -4 | cắt về 0: int(-7 / 2) = -3 | -(7 // 2) = -3


### 3.2 Prefix sum

`P[0] = 0`, `P[j] = nums[0] + … + nums[j−1]`. Tổng `nums[l..r]` (đóng) `= P[r + 1] − P[l]`.

In [3]:
nums = [7, 4, 3, 9, 1, 8, 5, 2, 6]
P = [0] + list(accumulate(nums))
print("P =", P)
print("tổng nums[0..6] =", P[7] - P[0], "| tổng nums[2..8] =", P[9] - P[2])

P = [0, 7, 11, 14, 23, 24, 32, 37, 39, 45]
tổng nums[0..6] = 37 | tổng nums[2..8] = 34


### 3.3 Cửa sổ trượt độ dài cố định

Giữ biến `s` = tổng cửa sổ hiện tại. Khi dịch sang phải một ô: `s += phần_tử_mới − phần_tử_rời_đi`.

In [4]:
w = 3; s = sum(nums[:w]); sums = [s]
for r in range(w, len(nums)):
    s += nums[r] - nums[r - w]; sums.append(s)
print("tổng mọi cửa sổ độ dài 3:", sums)
print("kiểm tra:", [sum(nums[i:i + w]) for i in range(len(nums) - w + 1)])

tổng mọi cửa sổ độ dài 3: [14, 16, 13, 18, 14, 15, 13]
kiểm tra: [14, 16, 13, 18, 14, 15, 13]


## 4. Ý tưởng nền tảng / trực giác

Với `k = 3`, cửa sổ quanh tâm `i` là `[i − 3, i + 3]`, độ dài `w = 7`. Tâm hợp lệ: `3 ≤ i ≤ 5` (vì `n = 9`, `i + 3 ≤ 8`).

| tâm `i` | cửa sổ | phần tử | tổng | `// 7` |
|---|---|---|---|---|
| 3 | [0, 6] | 7 4 3 9 1 8 5 | 37 | 5 |
| 4 | [1, 7] | 4 3 9 1 8 5 2 | 32 | 4 |
| 5 | [2, 8] | 3 9 1 8 5 2 6 | 34 | 4 |

**Quan sát:** cửa sổ tâm `4` = cửa sổ tâm `3` **bỏ** `7` (chỉ số 0) **thêm** `2` (chỉ số 7): `37 − 7 + 2 = 32`. Mỗi bước chỉ thay đổi hai phần tử ⇒ không cần cộng lại từ đầu.

In [5]:
nums, k = examples[0][0]; w = 2 * k + 1
for i in range(k, len(nums) - k):
    window = nums[i - k:i + k + 1]
    print(f"tâm {i}: cửa sổ {window}, tổng {sum(window)}, trung bình {sum(window) // w}")

tâm 3: cửa sổ [7, 4, 3, 9, 1, 8, 5], tổng 37, trung bình 5
tâm 4: cửa sổ [4, 3, 9, 1, 8, 5, 2], tổng 32, trung bình 4
tâm 5: cửa sổ [3, 9, 1, 8, 5, 2, 6], tổng 34, trung bình 4


Hình dung cửa sổ trượt trên ví dụ 1 (`w = 7`), dấu `[` `]` là biên cửa sổ, `^` là tâm:

```
chỉ số:    0 1 2 3 4 5 6 7 8
nums:      7 4 3 9 1 8 5 2 6
tâm 3:    [7 4 3 9 1 8 5]          tổng 37
                 ^
tâm 4:      [4 3 9 1 8 5 2]        tổng 37 - 7 + 2 = 32
                   ^
tâm 5:        [3 9 1 8 5 2 6]      tổng 32 - 4 + 6 = 34
                     ^
```

**Đếm số tâm hợp lệ:** tâm chạy từ `k` tới `n − 1 − k` ⇒ có `max(0, n − 2k)` tâm. Với `n = 9`, `k = 3`: `3` tâm. Mảng kết quả luôn có `min(k, n)` số `-1` ở mỗi đầu (khi `n ≤ 2k` thì toàn `-1`).

In [6]:
def count_valid_centers(n, k):
    return max(0, n - 2 * k)

for n_, k_ in [(9, 3), (1, 0), (1, 100000), (6, 3), (7, 3), (10, 0)]:
    got = sum(1 for i in range(n_) if i - k_ >= 0 and i + k_ < n_)
    print(f"n = {n_}, k = {k_}: số tâm hợp lệ = {count_valid_centers(n_, k_)} (đếm thật: {got})")

n = 9, k = 3: số tâm hợp lệ = 3 (đếm thật: 3)
n = 1, k = 0: số tâm hợp lệ = 1 (đếm thật: 1)
n = 1, k = 100000: số tâm hợp lệ = 0 (đếm thật: 0)
n = 6, k = 3: số tâm hợp lệ = 0 (đếm thật: 0)
n = 7, k = 3: số tâm hợp lệ = 1 (đếm thật: 1)
n = 10, k = 0: số tâm hợp lệ = 10 (đếm thật: 10)


## 5. Cách 1 — Brute Force / baseline: cộng lại từng cửa sổ

**Ý tưởng:** với mỗi `i`, nếu `i − k ≥ 0` và `i + k ≤ n − 1` thì cộng `2k + 1` phần tử bằng vòng lặp.

In [7]:
def averages_brute(nums, k):
    n = len(nums)
    res = [-1] * n
    for i in range(n):
        if i - k < 0 or i + k >= n:
            continue
        s = 0
        for j in range(i - k, i + k + 1):
            s += nums[j]
        res[i] = s // (2 * k + 1)
    return res

for (nums, k), want in examples:
    print(averages_brute(nums, k), "kỳ vọng", want)

[-1, -1, -1, 5, 4, 4, -1, -1, -1] kỳ vọng [-1, -1, -1, 5, 4, 4, -1, -1, -1]
[100000] kỳ vọng [100000]
[-1] kỳ vọng [-1]


**Độ phức tạp:** `O(n · (2k + 1))` thời gian. Tệ nhất khi `k ≈ n/4`: `≈ n²/4 = 2.5·10⁹` phép cộng với `n = 10⁵` — quá chậm. Space `O(n)` cho kết quả.

**Nút thắt:** hai cửa sổ liên tiếp chung `2k` phần tử nhưng ta cộng lại tất cả.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** chỉ cần **tổng** cửa sổ; tổng có thể lấy trong `O(1)` bằng prefix sum hoặc cập nhật trong `O(1)` khi trượt.

**Chỉ số then chốt** (với `w = 2k + 1`):
- Tâm `i` hợp lệ ⇔ `k ≤ i ≤ n − 1 − k` ⇔ cửa sổ `[i − k, i + k]` nằm trọn trong mảng.
- Prefix: `sum = P[i + k + 1] − P[i − k]`.
- Trượt: duyệt đầu phải `r = 0..n−1`, cộng `nums[r]`; khi `r ≥ w` bớt `nums[r − w]`; khi `r ≥ w − 1` cửa sổ đủ dài, tâm là `r − k`.

**Quy tắc biên:**
- *Nếu* `w > n` *thì* không có tâm hợp lệ ⇒ toàn `-1` (và tránh truy cập ngoài mảng).
- *Nếu* `k = 0` *thì* `w = 1`, kết quả là chính `nums`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix sum

In [8]:
def averages_prefix(nums, k):
    n, w = len(nums), 2 * k + 1
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    res = [-1] * n
    for i in range(k, n - k):                     # range rỗng nếu 2k + 1 > n
        res[i] = (P[i + k + 1] - P[i - k]) // w
    return res

for (nums, k), want in examples:
    print(averages_prefix(nums, k), "kỳ vọng", want)

[-1, -1, -1, 5, 4, 4, -1, -1, -1] kỳ vọng [-1, -1, -1, 5, 4, 4, -1, -1, -1]
[100000] kỳ vọng [100000]
[-1] kỳ vọng [-1]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ phụ cho `P`. Lưu ý `range(k, n − k)` tự rỗng khi `k ≥ n − k` ⇒ không cần `if` riêng cho trường hợp cửa sổ dài hơn mảng.

### 7.2 Cách 3 — Cửa sổ trượt (code nộp)

In [9]:
def averages_sliding(nums, k):
    n, w = len(nums), 2 * k + 1
    res = [-1] * n
    if w > n:
        return res
    s = 0
    for r, x in enumerate(nums):
        s += x                                    # phần tử mới vào cửa sổ
        if r >= w:
            s -= nums[r - w]                      # phần tử rời cửa sổ
        if r >= w - 1:
            res[r - k] = s // w                   # cửa sổ [r - 2k, r], tâm r - k
    return res

for (nums, k), want in examples:
    print(averages_sliding(nums, k), "kỳ vọng", want)

[-1, -1, -1, 5, 4, 4, -1, -1, -1] kỳ vọng [-1, -1, -1, 5, 4, 4, -1, -1, -1]
[100000] kỳ vọng [100000]
[-1] kỳ vọng [-1]


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ (ngoài mảng kết quả).

### 7.3 Cách 4 — `accumulate` + lát cắt (ít vòng lặp Python)

Tổng mọi cửa sổ độ dài `w` = `P[w:] − P[:-w]` từng phần tử. Ghép `k` số `-1` hai đầu.

In [10]:
def averages_vectorlike(nums, k):
    n, w = len(nums), 2 * k + 1
    if w > n:
        return [-1] * n
    P = [0, *accumulate(nums)]
    mids = [(b - a) // w for a, b in zip(P, P[w:])]   # len = n - w + 1
    return [-1] * k + mids + [-1] * k

for (nums, k), want in examples:
    print(averages_vectorlike(nums, k), "kỳ vọng", want)

[-1, -1, -1, 5, 4, 4, -1, -1, -1] kỳ vọng [-1, -1, -1, 5, 4, 4, -1, -1, -1]
[100000] kỳ vọng [100000]
[-1] kỳ vọng [-1]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Thường nhanh nhất trong Python vì `accumulate` và `zip` chạy trong C. Độ dài kết quả: `k + (n − 2k) + k = n` ✓.

### 7.4 (Tham khảo) Cửa sổ bằng `deque`

Dùng `deque` lưu các phần tử trong cửa sổ — hữu ích khi cần **nội dung** cửa sổ (min/max, trung vị), thừa với tổng nhưng là bước đệm cho các biến thể.

In [11]:
def averages_deque(nums, k):
    n, w = len(nums), 2 * k + 1
    res = [-1] * n; dq = deque(); s = 0
    for r, x in enumerate(nums):
        dq.append(x); s += x
        if len(dq) > w:
            s -= dq.popleft()
        if len(dq) == w:
            res[r - k] = s // w
    return res

for (nums, k), want in examples:
    print(averages_deque(nums, k), "kỳ vọng", want)

[-1, -1, -1, 5, 4, 4, -1, -1, -1] kỳ vọng [-1, -1, -1, 5, 4, 4, -1, -1, -1]
[100000] kỳ vọng [100000]
[-1] kỳ vọng [-1]


## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1: in `r`, phần tử vào/ra, tổng `s`, tâm được ghi.

In [12]:
def trace_sliding(nums, k):
    n, w = len(nums), 2 * k + 1; res = [-1] * n; s = 0
    print(f"w = {w}")
    print(f"{'r':>2} | vào | ra  | {'s':>3} | ghi")
    for r, x in enumerate(nums):
        s += x; out = "-"
        if r >= w:
            s -= nums[r - w]; out = nums[r - w]
        note = ""
        if r >= w - 1:
            res[r - k] = s // w; note = f"res[{r - k}] = {s} // {w} = {res[r - k]}"
        print(f"{r:>2} | {x:>3} | {str(out):>3} | {s:>3} | {note}")
    print("=>", res)

trace_sliding(*examples[0][0])

w = 7
 r | vào | ra  |   s | ghi
 0 |   7 |   - |   7 | 
 1 |   4 |   - |  11 | 
 2 |   3 |   - |  14 | 
 3 |   9 |   - |  23 | 
 4 |   1 |   - |  24 | 
 5 |   8 |   - |  32 | 
 6 |   5 |   - |  37 | res[3] = 37 // 7 = 5
 7 |   2 |   7 |  32 | res[4] = 32 // 7 = 4
 8 |   6 |   4 |  34 | res[5] = 34 // 7 = 4
=> [-1, -1, -1, 5, 4, 4, -1, -1, -1]


In [13]:
def trace_prefix(nums, k):
    n, w = len(nums), 2 * k + 1
    P = [0, *accumulate(nums)]
    print("P =", P)
    for i in range(k, n - k):
        print(f"  tâm {i}: P[{i + k + 1}] - P[{i - k}] = {P[i + k + 1]} - {P[i - k]} = {P[i + k + 1] - P[i - k]} -> // {w} = {(P[i + k + 1] - P[i - k]) // w}")

trace_prefix([2, 3, 1, 5, 0, 4], 1)

P = [0, 2, 5, 6, 11, 11, 15]
  tâm 1: P[3] - P[0] = 6 - 0 = 6 -> // 3 = 2
  tâm 2: P[4] - P[1] = 11 - 2 = 9 -> // 3 = 3
  tâm 3: P[5] - P[2] = 11 - 5 = 6 -> // 3 = 2
  tâm 4: P[6] - P[3] = 15 - 6 = 9 -> // 3 = 3


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def getAverages(self, nums: List[int], k: int) -> List[int]:
        n, w = len(nums), 2 * k + 1
        res = [-1] * n
        if w > n:
            return res
        s = 0
        for r, x in enumerate(nums):
            s += x
            if r >= w:
                s -= nums[r - w]
            if r >= w - 1:
                res[r - k] = s // w
        return res

sol = Solution()
for (nums, k), want in examples:
    got = sol.getAverages(nums, k)
    print(got, "OK" if got == want else "SAI")

[-1, -1, -1, 5, 4, 4, -1, -1, -1] OK
[100000] OK
[-1] OK


## 10. Vì sao thuật toán đúng

**Invariant (Cách 3).** Sau khi xử lý chỉ số `r`, `s = Σ nums[max(0, r − w + 1) .. r]` — tổng của (tối đa) `w` phần tử cuối.
- Khởi tạo `s = 0` (cửa sổ rỗng).
- Bước `r`: cộng `nums[r]`; nếu `r ≥ w` thì phần tử `nums[r − w]` vừa ra khỏi khoảng `[r − w + 1, r]` nên bị trừ. Invariant giữ.

Khi `r ≥ w − 1`, cửa sổ `[r − w + 1, r] = [r − 2k, r]` có đúng `w` phần tử, tâm `r − k`. Đây là **mọi** tâm hợp lệ: `r` chạy `w − 1..n − 1` ⇔ tâm chạy `k..n − 1 − k`. Các tâm khác giữ `-1`. ✓

**Cách 2.** `P[i + k + 1] − P[i − k] = Σ_{j = i−k}^{i+k} nums[j]` theo định nghĩa prefix; `range(k, n − k)` liệt kê đúng các tâm hợp lệ. ✓

**Cách 4.** `zip(P, P[w:])` ghép `P[j]` với `P[j + w]` cho `j = 0..n − w` ⇒ `n − w + 1 = n − 2k` cặp, mỗi cặp là tổng cửa sổ `[j, j + w − 1]` với tâm `j + k`. Ghép thêm `k` số `-1` mỗi đầu cho tổng độ dài `n`, và phần tử thứ `t` của `mids` rơi đúng vào vị trí `k + t` — tâm của cửa sổ thứ `t`. ✓

**Phép chia.** `nums[i] ≥ 0` ⇒ tổng `≥ 0`, `w > 0` ⇒ `s // w` = cắt phần thập phân. ✓

**Độ lớn:** tổng tối đa `n · 10⁵ = 10¹⁰` — vượt `int32` (cần `long` ở C++/Java), Python không sao.

## 11. Phân tích Time / Space

`n ≤ 10⁵`, `w = 2k + 1`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force | `O(n · w)` | `O(1)` |
| 2. Prefix | `O(n)` | `O(n)` |
| 3. Cửa sổ trượt | `O(n)` | `O(1)` |
| 4. `accumulate` + lát cắt | `O(n)` | `O(n)` |
| deque | `O(n)` | `O(w)` |

In [15]:
rng = random.Random(1)
big = [rng.randint(0, 10**5) for _ in range(10**5)]
for k in (0, 1000, 25000, 49999, 50000):
    row = []
    for name, fn in [("prefix", averages_prefix), ("sliding", averages_sliding), ("accumulate", averages_vectorlike),
                     ("deque", averages_deque), ("submit", Solution().getAverages)]:
        t0 = time.perf_counter(); r = fn(big, k); row.append((name, r, time.perf_counter() - t0))
    assert all(x[1] == row[0][1] for x in row)
    print(f"k = {k:>5}: " + ", ".join(f"{name} {t * 1000:.0f}ms" for name, _, t in row),
          f"| số tâm hợp lệ = {sum(v != -1 for v in row[0][1])}")
t0 = time.perf_counter(); averages_brute(big[:4000], 1000); print(f"brute n = 4000, k = 1000: {time.perf_counter() - t0:.2f}s")

k =     0: prefix 15ms, sliding 9ms, accumulate 10ms, deque 10ms, submit 10ms | số tâm hợp lệ = 100000


k =  1000: prefix 13ms, sliding 9ms, accumulate 8ms, deque 11ms, submit 10ms | số tâm hợp lệ = 98000


k = 25000: prefix 10ms, sliding 8ms, accumulate 6ms, deque 10ms, submit 8ms | số tâm hợp lệ = 50000


k = 49999: prefix 5ms, sliding 5ms, accumulate 2ms, deque 8ms, submit 5ms | số tâm hợp lệ = 2
k = 50000: prefix 5ms, sliding 0ms, accumulate 0ms, deque 8ms, submit 0ms | số tâm hợp lệ = 0


brute n = 4000, k = 1000: 0.09s


**Chia số thực có an toàn không?** Nhiều người viết `int(s / w)`. Với ràng buộc này, `s ≤ 10¹⁰ < 2⁵³` nên `s` và `w` biểu diễn chính xác bằng `float`, và phép chia `float` được làm tròn đúng (sai số ≤ nửa ulp của thương). Thương `≤ 10⁵` có ulp cỡ `1.5·10⁻¹¹`, còn khoảng cách từ một thương **không nguyên** tới số nguyên gần nhất ít nhất `1 / w ≥ 5·10⁻⁶` — lớn hơn sai số rất nhiều ⇒ `int(s / w) == s // w`. Nhưng khi số lớn hơn `2⁵³` thì không còn đúng, nên thói quen tốt là **luôn dùng `//`** với số nguyên.

In [16]:
rng = random.Random(3)
for _ in range(200000):
    w = 2 * rng.randint(0, 10**5) + 1
    s_ = rng.randint(0, 10**10)
    assert int(s_ / w) == s_ // w
big_s, w = 2**60 + 1, 3
print("trong ràng buộc: int(s / w) == s // w (200000 thử)")
print("ngoài ràng buộc, s = 2^60 + 1, w = 3:", int(big_s / w), "vs", big_s // w, "-> lệch", big_s // w - int(big_s / w))

trong ràng buộc: int(s / w) == s // w (200000 thử)
ngoài ràng buộc, s = 2^60 + 1, w = 3: 384307168202282304 vs 384307168202282325 -> lệch 21


## 12. Bộ test

In [17]:
fns = [averages_prefix, averages_sliding, averages_vectorlike, averages_deque, Solution().getAverages]

for (nums, k), want in examples:
    assert averages_brute(nums, k) == want
    for fn in fns:
        assert fn(nums, k) == want

edge = [
    (([1, 2, 3], 1), [-1, 2, -1]),                 # 2k + 1 = n: đúng một tâm
    (([1, 2], 1), [-1, -1]),                       # 2k + 1 > n
    (([5, 6, 7], 0), [5, 6, 7]),                   # k = 0
    (([0, 0, 0, 0, 0], 2), [-1, -1, 0, -1, -1]),
    (([1, 1, 1, 2], 1), [-1, 1, 1, -1]),           # 4 // 3 = 1
    (([10**5] * 5, 2), [-1, -1, 10**5, -1, -1]),
]
for (nums, k), want in edge:
    assert averages_brute(nums, k) == want, (nums, k)
    for fn in fns:
        assert fn(nums, k) == want, (fn, nums, k)

rng = random.Random(2024)
for _ in range(1000):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, rng.choice([1, 10, 10**5])) for _ in range(n)]
    k = rng.randint(0, n + 2)
    want = averages_brute(nums, k)
    for fn in fns:
        assert fn(nums, k) == want, (fn, nums, k)
print("ví dụ + edge + 1000 bộ ngẫu nhiên khớp brute force")

ví dụ + edge + 1000 bộ ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chia cho `2k` thay vì `2k + 1` (quên tính phần tử ở tâm)

In [18]:
def wrong_divisor(nums, k):
    n = len(nums); P = [0, *accumulate(nums)]; res = [-1] * n
    for i in range(k, n - k):
        res[i] = (P[i + k + 1] - P[i - k]) // max(1, 2 * k)     # SAI: cửa sổ có 2k + 1 phần tử
    return res

for (nums, k), want in examples[:2]:
    print("sai =", wrong_divisor(nums, k), "| đúng =", want)

sai = [-1, -1, -1, 6, 5, 5, -1, -1, -1] | đúng = [-1, -1, -1, 5, 4, 4, -1, -1, -1]
sai = [100000] | đúng = [100000]


Cửa sổ gồm `k` phần tử trái, `k` phần tử phải **và chính tâm**. **Sửa:** `w = 2k + 1`.

### Lỗi 2 — Lệch chỉ số prefix: `P[i + k] − P[i − k]`

In [19]:
def wrong_prefix_index(nums, k):
    n, w = len(nums), 2 * k + 1; P = [0, *accumulate(nums)]; res = [-1] * n
    for i in range(k, n - k):
        res[i] = (P[i + k] - P[i - k]) // w                    # SAI: thiếu nums[i + k]
    return res

for (nums, k), want in examples[:2]:
    print("sai =", wrong_prefix_index(nums, k), "| đúng =", want)

sai = [-1, -1, -1, 4, 4, 4, -1, -1, -1] | đúng = [-1, -1, -1, 5, 4, 4, -1, -1, -1]
sai = [0] | đúng = [100000]


`P[j]` là tổng **trước** chỉ số `j` ⇒ muốn gồm `nums[i + k]` phải dùng `P[i + k + 1]`. **Sửa:** `P[i + k + 1] − P[i − k]`.

### Lỗi 3 — Bớt sai phần tử khi trượt (`nums[r − w + 1]`)

In [20]:
def wrong_slide(nums, k):
    n, w = len(nums), 2 * k + 1; res = [-1] * n; s = 0
    if w > n: return res
    for r, x in enumerate(nums):
        s += x
        if r >= w:
            s -= nums[r - w + 1]                               # SAI: bớt phần tử vẫn còn trong cửa sổ
        if r >= w - 1:
            res[r - k] = s // w
    return res

for (nums, k), want in examples[:1] + [(([1, 2, 3, 4, 5], 1), [-1, 2, 3, 4, -1])]:
    print("sai =", wrong_slide(nums, k), "| đúng =", want)

sai = [-1, -1, -1, 5, 5, 5, -1, -1, -1] | đúng = [-1, -1, -1, 5, 4, 4, -1, -1, -1]
sai = [-1, 2, 2, 3, -1] | đúng = [-1, 2, 3, 4, -1]


Cửa sổ mới là `[r − w + 1, r]`; phần tử **rời đi** là `nums[r − w]`. **Sửa:** `s -= nums[r − w]`.

### Lỗi 4 — Không xử lý `2k + 1 > n` (truy cập ngoài mảng hoặc ghi sai chỗ)

In [21]:
def wrong_no_guard(nums, k):
    n, w = len(nums), 2 * k + 1; res = [-1] * n
    s = sum(nums[:w])                                          # lát cắt không báo lỗi khi w > n ...
    res_idx = k
    if res_idx < n:
        res[res_idx] = s // w                                  # SAI: ghi trung bình của cửa sổ thiếu phần tử
    for r in range(w, n):
        s += nums[r] - nums[r - w]; res[r - k] = s // w
    return res

for nums, k in [([4, 8], 1), ([9, 9, 9], 2), ([3], 0)]:
    print(f"nums = {nums}, k = {k}: sai = {wrong_no_guard(nums, k)} | đúng = {averages_brute(nums, k)}")

nums = [4, 8], k = 1: sai = [-1, 4] | đúng = [-1, -1]
nums = [9, 9, 9], k = 2: sai = [-1, -1, 5] | đúng = [-1, -1, -1]
nums = [3], k = 0: sai = [3] | đúng = [3]


Lát cắt `nums[:w]` với `w > n` **không** báo lỗi — trả về cả mảng — nên code "chạy được" nhưng ghi một giá trị bịa vào `res[k]` (khi `k < n`). **Sửa:** kiểm tra `w > n` ⇒ trả toàn `-1` trước.

### Lỗi 5 — Dùng `round` hoặc chia thực rồi làm tròn

In [22]:
def wrong_round(nums, k):
    n, w = len(nums), 2 * k + 1; P = [0, *accumulate(nums)]; res = [-1] * n
    for i in range(k, n - k):
        res[i] = round((P[i + k + 1] - P[i - k]) / w)          # SAI: đề cắt phần thập phân
    return res

for (nums, k), want in examples[:1] + [(([1, 1, 2], 1), [-1, 1, -1]), (([2, 3, 2, 3], 1), [-1, 2, 2, -1])]:
    print("sai =", wrong_round(nums, k), "| đúng =", want)

sai = [-1, -1, -1, 5, 5, 5, -1, -1, -1] | đúng = [-1, -1, -1, 5, 4, 4, -1, -1, -1]
sai = [-1, 1, -1] | đúng = [-1, 1, -1]
sai = [-1, 2, 3, -1] | đúng = [-1, 2, 2, -1]


`37 / 7 ≈ 5.29` làm tròn ra `5` (trùng may mắn), nhưng `32 / 7 ≈ 4.57` làm tròn ra `5` thay vì `4`. **Sửa:** `//` (với số không âm).

### Lỗi 6 — Ghi kết quả vào đầu phải `res[r]` thay vì tâm `res[r − k]`

In [23]:
def wrong_write_right_end(nums, k):
    n, w = len(nums), 2 * k + 1; res = [-1] * n; s = 0
    if w > n: return res
    for r, x in enumerate(nums):
        s += x
        if r >= w: s -= nums[r - w]
        if r >= w - 1:
            res[r] = s // w                                    # SAI: lệch k vị trí sang phải
    return res

for (nums, k), want in examples[:2] + [(([1, 2, 3, 4, 5], 1), [-1, 2, 3, 4, -1])]:
    print("sai =", wrong_write_right_end(nums, k), "| đúng =", want)

sai = [-1, -1, -1, -1, -1, -1, 5, 4, 4] | đúng = [-1, -1, -1, 5, 4, 4, -1, -1, -1]
sai = [100000] | đúng = [100000]
sai = [-1, -1, 2, 3, 4] | đúng = [-1, 2, 3, 4, -1]


Khi `k = 0` lỗi bị che (đầu phải trùng tâm) — vì vậy test chỉ với ví dụ 2 sẽ không phát hiện. **Sửa:** tâm của cửa sổ `[r − 2k, r]` là `r − k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng/trung bình của mọi đoạn độ dài cố định" ⇒ **cửa sổ trượt** hoặc prefix sum.
- Viết rõ ba mốc của cửa sổ trượt: **thêm** `nums[r]`, **bớt** `nums[r − w]` khi `r ≥ w`, **ghi** khi `r ≥ w − 1`.
- Tâm của cửa sổ `[r − 2k, r]` là `r − k`.
- Luôn test: `k = 0`, `2k + 1 = n`, `2k + 1 > n`, và một ví dụ có phần dư để thấy phép chia nguyên.
- Họ bài: 643 *Maximum Average Subarray I*, 1343 *Number of Sub-arrays of Size K and Average ≥ Threshold*, 1652 *Defuse the Bomb*, 2461 *Maximum Sum of Distinct Subarrays With Length K*.

## 15. Biến thể của bài

1. **Có số âm, chia "cắt về 0"**: `//` của Python làm tròn xuống ⇒ sai với tổng âm; dùng `int(s / w)` (cẩn thận độ chính xác số thực với số rất lớn) hoặc `-((-s) // w)` khi `s < 0`.
2. **Cửa sổ co lại ở biên** (thay vì `-1`, lấy trung bình của phần nằm trong mảng): cửa sổ `[max(0, i − k), min(n − 1, i + k)]`, chia cho độ dài thực.
3. **Trung vị / max của cửa sổ**: cần cấu trúc giữ nội dung cửa sổ (deque đơn điệu cho max, hai heap / `SortedList` cho trung vị).
4. **2D — làm mờ ảnh (box blur)**: trung bình hình vuông `(2k+1) × (2k+1)` quanh mỗi ô bằng prefix sum 2D (Bài 5; liên quan 1314 *Matrix Block Sum*, 661 *Image Smoother*).

In [24]:
def trunc_div(a, b):
    """Chia cắt về 0 cho số nguyên (b > 0), không dùng số thực."""
    return a // b if a >= 0 else -((-a) // b)

def averages_signed(nums, k):
    """Biến thể 1."""
    n, w = len(nums), 2 * k + 1; P = [0, *accumulate(nums)]; res = [-1] * n
    for i in range(k, n - k):
        res[i] = trunc_div(P[i + k + 1] - P[i - k], w)
    return res

def averages_shrinking(nums, k):
    """Biến thể 2: cửa sổ co ở biên (mọi vị trí đều có giá trị)."""
    n = len(nums); P = [0, *accumulate(nums)]
    return [(P[min(n - 1, i + k) + 1] - P[max(0, i - k)]) // (min(n - 1, i + k) - max(0, i - k) + 1) for i in range(n)]

from sortedcontainers import SortedList
def window_medians(nums, k):
    """Biến thể 3: trung vị (cửa sổ lẻ 2k+1 nên trung vị là phần tử thứ k)."""
    n, w = len(nums), 2 * k + 1; res = [-1] * n; sl = SortedList()
    for r, x in enumerate(nums):
        sl.add(x)
        if r >= w: sl.remove(nums[r - w])
        if r >= w - 1: res[r - k] = sl[k]
    return res

print("số âm:", averages_signed([-7, 0, 0], 1), "| // thường cho:", (-7) // 3, "| cắt về 0:", trunc_div(-7, 3))
print("co ở biên:", averages_shrinking([7, 4, 3, 9, 1, 8, 5, 2, 6], 3))
print("trung vị:", window_medians([7, 4, 3, 9, 1, 8, 5, 2, 6], 3))
rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(1, 15))]; k = rng.randint(0, 8)
    want = [-1 if i - k < 0 or i + k >= len(nums) else int(sum(nums[i - k:i + k + 1]) / (2 * k + 1)) for i in range(len(nums))]
    assert averages_signed(nums, k) == want
    want_med = [-1 if i - k < 0 or i + k >= len(nums) else sorted(nums[i - k:i + k + 1])[k] for i in range(len(nums))]
    assert window_medians(nums, k) == want_med
print("biến thể số âm và trung vị khớp brute force (biến thể tự đặt)")

số âm: [-1, -2, -1] | // thường cho: -3 | cắt về 0: -2
co ở biên: [5, 4, 5, 5, 4, 4, 5, 4, 5]
trung vị: [-1, -1, -1, 5, 4, 5, -1, -1, -1]
biến thể số âm và trung vị khớp brute force (biến thể tự đặt)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | cộng lại từng cửa sổ | `O(n w)` | `O(1)` | kiểm chứng |
| 2. Prefix | `P[i+k+1] − P[i−k]` | `O(n)` | `O(n)` | nhiều kiểu truy vấn đoạn |
| 3. Cửa sổ trượt | thêm/bớt | `O(n)` | `O(1)` | **nộp bài** |
| 4. `accumulate` + `zip` | prefix "vector" | `O(n)` | `O(n)` | nhanh nhất trong Python |
| deque | giữ nội dung | `O(n)` | `O(w)` | khi cần min/max/trung vị |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 2, 3, 4, 5, 6]`, `k = 1`? `k = 2`? `k = 3`?

**Bài 2 (tính tay):** `nums = [9, 0, 9, 0, 9, 0, 9]`, `k = 2`. Viết các cửa sổ, tổng và trung bình.

**Bài 3 (điền TODO):** hoàn thiện `window_sums(nums, w)` trả về list tổng của mọi cửa sổ độ dài `w` (độ dài `n − w + 1`; rỗng nếu `w > n`) bằng cửa sổ trượt.

**Bài 4 (tự viết lại):** viết `max_k_radius_average(nums, k)` trả về **giá trị lớn nhất** trong `avgs` (hoặc `-1` nếu không có tâm hợp lệ) mà không tạo mảng `avgs`.

**Bài 5 (mini challenge):** **box blur 2D** — cho lưới `grid` `m × n` số không âm và `k`; với mỗi ô có đủ `k` hàng/cột ở mọi phía, ghi trung bình (chia nguyên) của hình vuông `(2k+1) × (2k+1)` quanh nó, các ô khác `-1`. Dùng prefix sum 2D, `O(m n)`. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [25]:
def window_sums(nums, w):
    # TODO: s += nums[r]; nếu r >= w thì s -= nums[r - w]; nếu r >= w - 1 thì append(s)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `k = 1`: cửa sổ 3 phần tử, tâm 1..4. (2) Trung bình của 3 số liên tiếp là số giữa. (3) `[-1,2,3,4,5,-1]`; `k = 2`: `[-1,-1,3,4,-1,-1]`; `k = 3`: `w = 7 > 6` ⇒ toàn `-1`.

**Bài 2:** (1) `w = 5`, tâm 2, 3, 4. (2) Tổng: `27`, `18`, `27`. (3) `27 // 5 = 5`, `18 // 5 = 3` ⇒ `[-1,-1,5,3,5,-1,-1]`.

**Bài 3:** (1) Như Cách 3 nhưng `append` thay vì ghi vào tâm. (2) Kiểm tra `w > n` ⇒ `[]`. (3) So với `[sum(nums[i:i+w]) ...]`.

**Bài 4:** (1) Trượt cửa sổ, giữ `best = max(best, s)`. (2) Chia **một lần** ở cuối: `best // w` (vì `//` với số chia dương là hàm không giảm). (3) Nếu `w > n` ⇒ `-1`.

**Bài 5:** (1) `S[i][j]` = tổng hình chữ nhật `[0, i) × [0, j)`. (2) Tổng hình vuông `[r1, r2] × [c1, c2]` = `S[r2+1][c2+1] − S[r1][c2+1] − S[r2+1][c1] + S[r1][c1]`. (3) Ô hợp lệ: `k ≤ i < m − k`, `k ≤ j < n − k`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().getAverages([1, 2, 3, 4, 5, 6], 1) == [-1, 2, 3, 4, 5, -1]
assert Solution().getAverages([1, 2, 3, 4, 5, 6], 2) == [-1, -1, 3, 4, -1, -1]
assert Solution().getAverages([1, 2, 3, 4, 5, 6], 3) == [-1] * 6
# Bài 2
assert Solution().getAverages([9, 0, 9, 0, 9, 0, 9], 2) == averages_brute([9, 0, 9, 0, 9, 0, 9], 2) == [-1, -1, 5, 3, 5, -1, -1]

# Bài 3
def window_sums_ans(nums, w):
    out, s = [], 0
    for r, x in enumerate(nums):
        s += x
        if r >= w: s -= nums[r - w]
        if r >= w - 1: out.append(s)
    return out

# Bài 4
def max_k_radius_average_ans(nums, k):
    w = 2 * k + 1
    sums = window_sums_ans(nums, w)
    return max(sums) // w if sums else -1

rng = random.Random(41)
for _ in range(500):
    nums = [rng.randint(0, 50) for _ in range(rng.randint(1, 20))]; w = rng.randint(1, 22); k = rng.randint(0, 10)
    assert window_sums_ans(nums, w) == [sum(nums[i:i + w]) for i in range(len(nums) - w + 1)]
    valid = [v for v in averages_brute(nums, k) if v != -1]
    assert max_k_radius_average_ans(nums, k) == (max(valid) if valid else -1)
print("Bài 4 ví dụ 1:", max_k_radius_average_ans([7, 4, 3, 9, 1, 8, 5, 2, 6], 3))
print("Bài 1-4: OK")

Bài 4 ví dụ 1: 5
Bài 1-4: OK


In [27]:
# Bài 5 — box blur 2D bằng prefix sum 2D
def box_blur(grid, k):
    m, n = len(grid), len(grid[0]); w = (2 * k + 1) ** 2
    S = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            S[i + 1][j + 1] = grid[i][j] + S[i][j + 1] + S[i + 1][j] - S[i][j]
    res = [[-1] * n for _ in range(m)]
    for i in range(k, m - k):
        for j in range(k, n - k):
            r1, c1, r2, c2 = i - k, j - k, i + k, j + k
            tot = S[r2 + 1][c2 + 1] - S[r1][c2 + 1] - S[r2 + 1][c1] + S[r1][c1]
            res[i][j] = tot // w
    return res

def box_blur_brute(grid, k):
    m, n = len(grid), len(grid[0]); res = [[-1] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if k <= i < m - k and k <= j < n - k:
                vals = [grid[a][b] for a in range(i - k, i + k + 1) for b in range(j - k, j + k + 1)]
                res[i][j] = sum(vals) // len(vals)
    return res

rng = random.Random(55)
for _ in range(200):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = [[rng.randint(0, 30) for _ in range(n)] for _ in range(m)]; k = rng.randint(0, 3)
    assert box_blur(g, k) == box_blur_brute(g, k)
demo = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
print("Bài 5 ví dụ, k = 1:")
for row in box_blur(demo, 1): print("  ", row)

Bài 5 ví dụ, k = 1:
   [-1, -1, -1, -1]
   [-1, 6, 7, -1]
   [-1, 10, 11, -1]
   [-1, -1, -1, -1]


Ghi chú cho Bài 4: vì sao được chia **một lần** ở cuối? Hàm `x ↦ x // w` (với `w > 0`) **không giảm**: `a ≤ b ⇒ a // w ≤ b // w`. Nên `max(s_i // w) = (max s_i) // w`. Tính chất này giúp tránh `n` phép chia — mẹo nhỏ nhưng hay dùng khi so sánh trung bình các cửa sổ **cùng độ dài** (bài 643 *Maximum Average Subarray I* so sánh tổng thay vì trung bình).

Ghi chú cho Bài 5: số ô hợp lệ là `max(0, m − 2k) · max(0, n − 2k)`; mỗi ô tốn `O(1)` nhờ công thức bao hàm–loại trừ 4 góc. Dựng `S` tốn `O(m n)`. Đây đúng là phiên bản 2D của Cách 2.

## 20. Template nhớ nhanh + Checklist

```python
# Tổng mọi cửa sổ độ dài w = 2k + 1, ghi vào tâm
n, w = len(nums), 2 * k + 1
res = [-1] * n
if w > n: return res
s = 0
for r, x in enumerate(nums):
    s += x                       # vào
    if r >= w: s -= nums[r - w]  # ra
    if r >= w - 1: res[r - k] = s // w   # đủ dài -> tâm r - k
# Prefix: res[i] = (P[i + k + 1] - P[i - k]) // w  với k <= i <= n - 1 - k
```

**Checklist trước khi submit:**
- [ ] Độ dài cửa sổ `2k + 1` (gồm tâm).
- [ ] `2k + 1 > n` ⇒ toàn `-1`, không truy cập ngoài mảng.
- [ ] Bớt `nums[r − w]`, ghi vào `res[r − k]`.
- [ ] Prefix: `P[i + k + 1] − P[i − k]`.
- [ ] Chia nguyên `//` (đúng vì số không âm); tổng có thể tới `10¹⁰`.